# Install & Import Library

In [49]:
%pip install torch
%pip install transformers
%pip install conllu
%pip install pandas numpy scikit-learn matplotlib tqdm
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import conllu
import os
import pandas as pd
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display
from collections import defaultdict
import openpyxl

# Explore Dataset IndoLEM

## A. Explore Task Dataset - UD_Indonesian_GSD (RAW)

### 1. Set Path

In [59]:
BASE_PATH = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\raw\indolem-main\dependency_parsing"

PATH_GSD_TRAIN = os.path.join(BASE_PATH, "UD_Indonesian_GSD", "id_gsd-ud-train.conllu")
PATH_GSD_DEV   = os.path.join(BASE_PATH, "UD_Indonesian_GSD", "id_gsd-ud-dev.conllu")
PATH_GSD_TEST  = os.path.join(BASE_PATH, "UD_Indonesian_GSD", "id_gsd-ud-test.conllu")

### 2. Exploration Raw Data

In [60]:
# ============================================
# BACA DATA GSD
# ============================================
with open(PATH_GSD_TRAIN, "r", encoding="utf-8") as f:
    data_gsd_train = conllu.parse(f.read())
with open(PATH_GSD_DEV, "r", encoding="utf-8") as f:
    data_gsd_dev = conllu.parse(f.read())
with open(PATH_GSD_TEST, "r", encoding="utf-8") as f:
    data_gsd_test = conllu.parse(f.read())

data_gsd_all = data_gsd_train + data_gsd_dev + data_gsd_test

# ============================================
# AMBIL SAMPLE RAW DATA DARI GSD
# ============================================
raw_data = []

for kalimat in data_gsd_all:
    teks = kalimat.metadata.get('text', '')
    sent_id = kalimat.metadata.get('sent_id', '')
    
    for token in kalimat:
        if not isinstance(token['id'], int):
            continue
            
        raw_data.append({
            'sent_id'  : sent_id,
            'teks'     : teks,
            'token_id' : token['id'],
            'token'    : token['form'],
            'lemma'    : token['lemma'],
            'upos'     : token['upos'],
            'xpos'     : token['xpos'],
            'feats'    : str(token['feats']) if token['feats'] else '-',
            'head'     : token['head'],
            'deprel'   : token['deprel'],
            'misc'     : str(token['misc']) if token['misc'] else '-',
        })

df_raw = pd.DataFrame(raw_data)

# ============================================
# TAMPILKAN STATISTIK LABEL
# ============================================
print("=" * 60)
print("INFORMASI RAW DATA UD-INDO-GSD")
print("=" * 60)

print(f"\n{'Jumlah kalimat (all)':<30}: {len(data_gsd_all)}")
print(f"{'Jumlah kalimat (train)':<30}: {len(data_gsd_train)}")
print(f"{'Jumlah kalimat (dev)':<30}: {len(data_gsd_dev)}")
print(f"{'Jumlah kalimat (test)':<30}: {len(data_gsd_test)}")

# Hitung total token
total_token = sum(
    1 for kalimat in data_gsd_all
    for token in kalimat
    if isinstance(token['id'], int)
)
total_token_no_punct = sum(
    1 for kalimat in data_gsd_all
    for token in kalimat
    if isinstance(token['id'], int) and token['upos'] != 'PUNCT'
)

# HITUNG TOTAL TOKEN PER SPLIT + ALL
def hitung_token(data, include_punct=True):
    return sum(
        1 for kalimat in data
        for token in kalimat
        if isinstance(token['id'], int)
        and (include_punct or token['upos'] != 'PUNCT')
    )

splits = {
    'train' : data_gsd_train,
    'dev'   : data_gsd_dev,
    'test'  : data_gsd_test,
    'all'   : data_gsd_all,
}

print(f"\n{'Split':<10} {'Token (incl. PUNCT)':>22} {'Token (excl. PUNCT)':>22}")
print("-" * 56)
for nama, data in splits.items():
    total        = hitung_token(data, include_punct=True)
    total_no_punc = hitung_token(data, include_punct=False)
    print(f"{nama:<10} {total:>22,} {total_no_punc:>22,}")

# ============================================
# TABEL 1: FIELD YANG TERSEDIA
# ============================================
print("\n\n=== TABEL: FIELD YANG TERSEDIA ===")
fields_info = {
    'sent_id' : 'ID kalimat',
    'token'   : 'Bentuk token asli',
    'lemma'   : 'Bentuk dasar token',
    'upos'    : 'Universal POS tag',
    'xpos'    : 'Language-specific POS tag',
    'feats'   : 'Morphological features',
    'head'    : 'ID token kepala (dependency)',
    'deprel'  : 'Dependency relation label',
    'misc'    : 'Informasi tambahan (MorphInd)',
}

df_fields = pd.DataFrame([
    {'FIELD': k, 'KETERANGAN': v}
    for k, v in fields_info.items()
])
display(df_fields)

# ============================================
# TABEL 2: DISTRIBUSI UPOS
# ============================================
print("\n\n=== TABEL: DISTRIBUSI UPOS ===")
upos_data = []
for kalimat in data_gsd_all:
    for token in kalimat:
        if isinstance(token['id'], int) and token['upos'] != 'PUNCT':
            upos_data.append(token['upos'])

upos_counter = Counter(upos_data)
total = sum(upos_counter.values())

df_upos = pd.DataFrame([
    {
        'UPOS'       : upos,
        'JUMLAH'     : count,
        'KETERANGAN' : {
            'NOUN' : 'Kata benda',
            'VERB' : 'Kata kerja',
            'ADJ'  : 'Kata sifat',
            'ADV'  : 'Kata keterangan',
            'PROPN': 'Kata benda proper',
            'DET'  : 'Determiner',
            'ADP'  : 'Adposisi',
            'PRON' : 'Kata ganti',
            'CCONJ': 'Konjungsi koordinatif',
            'SCONJ': 'Konjungsi subordinatif',
            'NUM'  : 'Numeralia',
            'AUX'  : 'Kata bantu',
            'PART' : 'Partikel',
            'INTJ' : 'Interjeksi',
            'SYM'  : 'Simbol',
            'X'    : 'Lainnya',
        }.get(upos, '-')
    }
    for upos, count in upos_counter.most_common()
])
display(df_upos)

# ============================================
# TABEL 3: DISTRIBUSI XPOS
# ============================================

# KETERANGAN XPOS (Language-Specific POS Tag - MorphInd/IDT tagset)
XPOS_KETERANGAN = {
    # ── NOMINA (N) ──────────────────────────────────────────────────
    'NSD'     : 'Nomina dasar (common noun)',
    'NSD--'   : 'Nomina dasar tanpa fitur morfologi',
    'NSD-SG'  : 'Nomina dasar bentuk tunggal',
    'NSD-PL'  : 'Nomina dasar bentuk jamak (reduplikasi)',
    'NSD-SG-' : 'Nomina dasar tunggal tanpa fitur tambahan',
    'NSD-PL-' : 'Nomina dasar jamak tanpa fitur tambahan',
    'NSP'     : 'Nomina proper (nama diri)',
    'NSP--'   : 'Nomina proper tanpa fitur morfologi',
    'NSP-SG'  : 'Nomina proper bentuk tunggal',
    'NSP-PL'  : 'Nomina proper bentuk jamak',

    # ── VERBA (V) ────────────────────────────────────────────────────
    'VS--'    : 'Verba dasar tanpa fitur (bare verb)',
    'VSA'     : 'Verba aktif (berafiks me-)',
    'VSA-'    : 'Verba aktif tanpa fitur tambahan',
    'VSA--'   : 'Verba aktif tanpa fitur morfologi',
    'VSAP'    : 'Verba aktif-pasif (dapat aktif dan pasif)',
    'VSP'     : 'Verba pasif (berafiks di-/ter-)',
    'VSP-'    : 'Verba pasif tanpa fitur tambahan',
    'VSP--'   : 'Verba pasif tanpa fitur morfologi',
    'VSB'     : 'Verba benefaktif (berafiks memper-/diper-)',
    'VSB-'    : 'Verba benefaktif tanpa fitur tambahan',
    'VSD'     : 'Verba dasar (stative/intransitive)',
    'VSD-'    : 'Verba dasar tanpa fitur tambahan',

    # ── ADJEKTIVA (A) ────────────────────────────────────────────────
    'AS--'    : 'Adjektiva tanpa fitur morfologi',
    'ASP'     : 'Adjektiva positif (derajat biasa)',
    'ASP-'    : 'Adjektiva positif tanpa fitur tambahan',
    'ASC'     : 'Adjektiva komparatif (lebih ...)',
    'ASC-'    : 'Adjektiva komparatif tanpa fitur tambahan',
    'ASS'     : 'Adjektiva superlatif (paling ...)',
    'ASS-'    : 'Adjektiva superlatif tanpa fitur tambahan',

    # ── ADVERBIA (D) ─────────────────────────────────────────────────
    'D--'     : 'Adverbia umum',
    'D---'    : 'Adverbia tanpa fitur morfologi',
    'D-F'     : 'Adverbia frekuensi (selalu, sering, jarang)',
    'D-T'     : 'Adverbia temporal (kemarin, besok, sudah)',
    'D-M'     : 'Adverbia modalitas (mungkin, tentu, pasti)',
    'D-N'     : 'Adverbia negasi (tidak, bukan, belum, jangan)',

    # ── PREPOSISI / ADPOSISI (R) ─────────────────────────────────────
    'R--'     : 'Preposisi umum (di, ke, dari, dengan)',
    'R---'    : 'Preposisi tanpa fitur morfologi',

    # ── KONJUNGSI KOORDINATIF (CC) ───────────────────────────────────
    'CC--'    : 'Konjungsi koordinatif (dan, atau, tetapi)',
    'CC---'   : 'Konjungsi koordinatif tanpa fitur morfologi',

    # ── KONJUNGSI SUBORDINATIF (SC) ──────────────────────────────────
    'SC--'    : 'Konjungsi subordinatif (bahwa, karena, jika, meski)',
    'SC---'   : 'Konjungsi subordinatif tanpa fitur morfologi',

    # ── PRONOMINA (P) ────────────────────────────────────────────────
    'PS1'     : 'Pronomina persona pertama (saya, aku, kami, kita)',
    'PS1-SG'  : 'Pronomina persona pertama tunggal (saya, aku)',
    'PS1-PL'  : 'Pronomina persona pertama jamak (kami, kita)',
    'PS2'     : 'Pronomina persona kedua (kamu, Anda, kalian)',
    'PS2-SG'  : 'Pronomina persona kedua tunggal (kamu, Anda)',
    'PS2-PL'  : 'Pronomina persona kedua jamak (kalian)',
    'PS3'     : 'Pronomina persona ketiga (dia, ia, mereka)',
    'PS3-SG'  : 'Pronomina persona ketiga tunggal (dia, ia)',
    'PS3-PL'  : 'Pronomina persona ketiga jamak (mereka)',
    'PD--'    : 'Pronomina demonstratif (ini, itu, sini, situ)',
    'PI--'    : 'Pronomina interogatif (apa, siapa, mana)',
    'PR--'    : 'Pronomina relatif (yang)',
    'PIN-'    : 'Pronomina indefinit (sesuatu, seseorang, masing-masing)',
    'PN--'    : 'Pronomina negatif (tidak seorang pun, tidak ada)',

    # ── NUMERALIA (Q) ────────────────────────────────────────────────
    'Q--'     : 'Numeralia kardinal (satu, dua, 10, 100)',
    'Q---'    : 'Numeralia kardinal tanpa fitur morfologi',
    'QO-'     : 'Numeralia ordinal (pertama, kedua, ke-3)',
    'QO--'    : 'Numeralia ordinal tanpa fitur tambahan',
    'QF-'     : 'Numeralia fraksional (setengah, seperempat)',
    'QC-'     : 'Numeralia kolektif (kedua-duanya, bertiga)',

    # ── DETERMINER (T) ───────────────────────────────────────────────
    'T--'     : 'Determiner/penentu (para, sang, si, kaum)',
    'T---'    : 'Determiner tanpa fitur morfologi',

    # ── KATA BANTU / AUXILIARI (O) ───────────────────────────────────
    'O--'     : 'Kata bantu (akan, sedang, telah, sudah, dapat, harus)',
    'O---'    : 'Kata bantu tanpa fitur morfologi',

    # ── PARTIKEL (G) ─────────────────────────────────────────────────
    'G--'     : 'Partikel penegas (-lah, -kah, -pun, toh)',
    'G---'    : 'Partikel tanpa fitur morfologi',

    # ── KATA SERU / INTERJEKSI (I) ───────────────────────────────────
    'I--'     : 'Interjeksi / kata seru (wah, aduh, hei, ya)',

    # ── KATA ASING (F) ───────────────────────────────────────────────
    'F--'     : 'Kata asing (foreign word, belum diserap)',

    # ── TANDA BACA / SIMBOL (Z) ──────────────────────────────────────
    'Z--'     : 'Tanda baca (titik, koma, tanda tanya)',
    'Z---'    : 'Tanda baca tanpa fitur morfologi',

    # ── TIDAK TERKLASIFIKASI (X) ─────────────────────────────────────
    'X--'     : 'Kategori tidak terklasifikasi / kata asing tak dikenal',
    'X---'    : 'Tidak terklasifikasi tanpa fitur morfologi',
    '_'       : 'Tidak ada xpos (token tanpa anotasi xpos)',
}

print("\n\n=== TABEL: DISTRIBUSI XPOS ===")

xpos_data = []
for kalimat in data_gsd_all:
    for token in kalimat:
        if isinstance(token['id'], int) and token['upos'] != 'PUNCT':
            xpos_data.append(token['xpos'])

xpos_counter = Counter(xpos_data)
total_xpos   = sum(xpos_counter.values())

df_xpos = pd.DataFrame([
    {
        'XPOS'       : xpos,
        'JUMLAH'     : count,
        'KETERANGAN' : XPOS_KETERANGAN.get(xpos, '-'),
    }
    for xpos, count in xpos_counter.most_common()
])
display(df_xpos)

# ============================================
# TABEL 4: DISTRIBUSI DEPREL
# ============================================

# KETERANGAN DEPREL
DEPREL_KETERANGAN = {
    # Core arguments
    'root'       : 'Kata utama kalimat',
    'nsubj'      : 'Subjek nominal',
    'nsubj:pass' : 'Subjek nominal kalimat pasif',
    'obj'        : 'Objek langsung',
    'iobj'       : 'Objek tidak langsung',
    'csubj'      : 'Subjek klausal',
    'csubj:pass' : 'Subjek klausal kalimat pasif',
    'ccomp'      : 'Komplemen klausal',
    'xcomp'      : 'Komplemen klausal terbuka',

    # Nominal dependents
    'nmod'       : 'Modifier nominal',
    'nmod:poss'  : 'Modifier posesif',
    'appos'      : 'Aposisi',
    'nummod'     : 'Modifier numerik',
    'amod'       : 'Modifier adjektival',
    'det'        : 'Determiner',
    'clf'        : 'Classifier',
    'case'       : 'Preposisi/postposisi',

    # Verb dependents
    'obl'        : 'Keterangan oblique',
    'obl:agent'  : 'Keterangan agen (pasif)',
    'advmod'     : 'Modifier adverbial',
    'aux'        : 'Kata bantu',
    'aux:pass'   : 'Kata bantu pasif',
    'cop'        : 'Kopula',
    'compound'   : 'Kata majemuk',
    'compound:plur' : 'Reduplikasi (kata ulang jamak)',
    'flat'       : 'Ekspresi flat (nama)',
    'flat:name'  : 'Nama diri',
    'fixed'      : 'Ekspresi tetap',

    # Coordination
    'conj'       : 'Konjungsi koordinatif',
    'cc'         : 'Konjungsi koordinatif (kata)',
    'cc:preconj' : 'Konjungsi pre-koordinatif',

    # Clausal dependents
    'acl'        : 'Klausa adjektival',
    'acl:relcl'  : 'Klausa relatif',
    'advcl'      : 'Klausa adverbial',
    'mark'       : 'Penanda klausa (bahwa/yang)',
    'parataxis'  : 'Hubungan parataktis',
    'list'       : 'Hubungan list',
    'orphan'     : 'Orphan (ellipsis)',
    'reparandum' : 'Koreksi ujaran',
    'discourse'  : 'Penanda wacana',

    # Special
    'punct'      : 'Tanda baca',
    'dep'        : 'Dependensi tidak terspesifikasi',
    'goeswith'   : 'Bagian dari kata yang sama',
    'vocative'   : 'Vokativa',
    'expl'       : 'Ekspletif',
    'dislocated' : 'Dislokasi',
}


print("=== TABEL DISTRIBUSI DEPREL ===")

deprel_data = []
for kalimat in data_gsd_all:
    for token in kalimat:
        if isinstance(token['id'], int) and token['upos'] != 'PUNCT':
            deprel_data.append(token['deprel'])

deprel_counter = Counter(deprel_data)
total_deprel   = sum(deprel_counter.values())

df_deprel = pd.DataFrame([
    {
        'DEPREL'     : deprel,
        'JUMLAH'     : count,
        'KETERANGAN' : DEPREL_KETERANGAN.get(deprel, '-')
    }
    for deprel, count in deprel_counter.most_common()
])
display(df_deprel)


# ============================================
# TABEL 5: DISTRIBUSI FEATS
# ============================================

# KETERANGAN MORPHOLOGICAL FEATURES

FEATS_KETERANGAN = {
    # Voice
    'Voice=Act'         : 'Verba aktif (me-)',
    'Voice=Pass'        : 'Verba pasif (di-)',

    # Number
    'Number=Sing'       : 'Bentuk tunggal',
    'Number=Plur'       : 'Bentuk jamak',

    # PronType
    'PronType=Rel'      : 'Pronomina relatif (yang)',
    'PronType=Dem'      : 'Pronomina demonstratif (ini/itu)',
    'PronType=Ind'      : 'Pronomina indefinit (sebuah)',
    'PronType=Prs'      : 'Pronomina persona (saya/dia)',
    'PronType=Int'      : 'Pronomina interogatif (apa/siapa)',
    'PronType=Tot'      : 'Pronomina total (semua)',
    'PronType=Neg'      : 'Pronomina negatif (tidak ada)',

    # Degree
    'Degree=Pos'        : 'Adjektiva positif',
    'Degree=Cmp'        : 'Adjektiva komparatif (lebih)',
    'Degree=Sup'        : 'Adjektiva superlatif (paling)',

    # NumType
    'NumType=Card'      : 'Numeralia kardinal (satu, dua)',
    'NumType=Ord'       : 'Numeralia ordinal (pertama, kedua)',
    'NumType=Frac'      : 'Numeralia pecahan (setengah)',
    'NumType=Mult'      : 'Numeralia multiplikatif (dua kali)',
    'NumType=Dist'      : 'Numeralia distributif',
    'NumType=Sets'      : 'Numeralia set',

    # Person
    'Person=1'          : 'Persona pertama (saya/kami)',
    'Person=2'          : 'Persona kedua (kamu/Anda)',
    'Person=3'          : 'Persona ketiga (dia/mereka)',

    # Polarity
    'Polarity=Neg'      : 'Negasi (tidak/bukan/belum)',
    'Polarity=Pos'      : 'Positif (afirmatif)',

    # Possessor
    'Number[psor]=Sing' : 'Pemilik tunggal (-nya/-ku/-mu)',
    'Number[psor]=Plur' : 'Pemilik jamak',
    'Person[psor]=1'    : 'Pemilik persona pertama (-ku)',
    'Person[psor]=2'    : 'Pemilik persona kedua (-mu)',
    'Person[psor]=3'    : 'Pemilik persona ketiga (-nya)',

    # Polite
    'Polite=Form'       : 'Bentuk formal/hormat (Anda)',
    'Polite=Infm'       : 'Bentuk informal (kamu)',

    # Clusivity
    'Clusivity=Ex'      : 'Eksklusif (kami — tidak termasuk lawan bicara)',
    'Clusivity=In'      : 'Inklusif (kita — termasuk lawan bicara)',

    # Gender
    'Gender=Masc'       : 'Maskulin',
    'Gender=Fem'        : 'Feminin',

    # Mood
    'Mood=Imp'          : 'Modus imperatif (perintah)',
    'Mood=Ind'          : 'Modus indikatif (pernyataan)',
    'Mood=Sub'          : 'Modus subjunktif',

    # Aspect
    'Aspect=Perf'       : 'Aspek perfektif (sudah)',
    'Aspect=Imp'        : 'Aspek imperfektif (sedang)',
    'Aspect=Iter'       : 'Aspek iteratif (berulang)',
}

print("\n\n=== TABEL DISTRIBUSI MORPHOLOGICAL FEATURES ===")

feats_data = []
for kalimat in data_gsd_all:
    for token in kalimat:
        if not isinstance(token['id'], int):
            continue
        if token['upos'] == 'PUNCT':
            continue
        if token['feats']:
            for key, val in token['feats'].items():
                feats_data.append(f"{key}={val}")

feats_counter = Counter(feats_data)
total_feats   = sum(feats_counter.values())

df_feats = pd.DataFrame([
    {
        'FEATURE'    : feat,
        'JUMLAH'     : count,
        'KETERANGAN' : FEATS_KETERANGAN.get(feat, '-')
    }
    for feat, count in feats_counter.most_common()
])
display(df_feats)


# ============================================
# TABEL 6: SAMPLE RAW DATA LENGKAP
# ============================================
print("\n\n=== TABEL: SAMPLE RAW DATA (20 baris pertama) ===")
display(df_raw[['sent_id', 'teks', 'token', 'lemma', 'upos', 'xpos',
                'feats', 'head', 'deprel', 'misc']].head(20))

INFORMASI RAW DATA UD-INDO-GSD

Jumlah kalimat (all)          : 5593
Jumlah kalimat (train)        : 4477
Jumlah kalimat (dev)          : 559
Jumlah kalimat (test)         : 557

Split         Token (incl. PUNCT)    Token (excl. PUNCT)
--------------------------------------------------------
train                      97,531                 82,963
dev                        12,612                 10,676
test                       11,780                 10,056
all                       121,923                103,695


=== TABEL: FIELD YANG TERSEDIA ===


,FIELD,KETERANGAN
0,sent_id,ID kalimat
1,token,Bentuk token asli
2,lemma,Bentuk dasar token
3,upos,Universal POS tag
4,xpos,Language-specific POS tag
5,feats,Morphological features
6,head,ID token kepala (dependency)
7,deprel,Dependency relation label
8,misc,Informasi tambahan (MorphInd)




=== TABEL: DISTRIBUSI UPOS ===


,UPOS,JUMLAH,KETERANGAN
0,NOUN,27000,Kata benda
1,PROPN,22790,Kata benda proper
2,VERB,12202,Kata kerja
3,ADP,12019,Adposisi
4,PRON,4764,Kata ganti
5,ADV,4760,Kata keterangan
6,ADJ,4528,Kata sifat
7,NUM,4383,Numeralia
8,DET,4012,Determiner
9,CCONJ,3659,Konjungsi koordinatif




=== TABEL: DISTRIBUSI XPOS ===


...

,DEPREL,JUMLAH,KETERANGAN
0,case,11897,Preposisi/postposisi
1,flat,11402,Ekspresi flat (nama)
2,compound,7428,Kata majemuk
3,nsubj,7125,Subjek nominal
4,obl,6346,Keterangan oblique
5,obj,5794,Objek langsung
6,root,5592,Kata utama kalimat
7,advmod,5288,Modifier adverbial
8,conj,4806,Konjungsi koordinatif
9,amod,4566,Modifier adjektival




=== TABEL DISTRIBUSI MORPHOLOGICAL FEATURES ===


,FEATURE,JUMLAH,KETERANGAN
0,Number=Sing,49801,Bentuk tunggal
1,Voice=Act,9352,Verba aktif (me-)
2,Degree=Pos,6001,Adjektiva positif
3,NumType=Card,4714,"Numeralia kardinal (satu, dua)"
4,Voice=Pass,3527,Verba pasif (di-)
5,PronType=Rel,3110,Pronomina relatif (yang)
6,PronType=Dem,2048,Pronomina demonstratif (ini/itu)
7,Number[psor]=Sing,1877,Pemilik tunggal (-nya/-ku/-mu)
8,Person[psor]=3,1785,Pemilik persona ketiga (-nya)
9,PronType=Ind,1138,Pronomina indefinit (sebuah)




=== TABEL: SAMPLE RAW DATA (20 baris pertama) ===


,sent_id,teks,token,lemma,upos,xpos,feats,head,deprel,misc
0,train-s1,Sembungan adalah sebuah desa yang terletak di ...,Sembungan,sembungan,PROPN,X--,-,4,nsubj,{'MorphInd': '^sembungan<x>_X--$'}
1,train-s1,Sembungan adalah sebuah desa yang terletak di ...,adalah,adalah,AUX,O--,-,4,cop,{'MorphInd': '^adalah<o>_O--$'}
2,train-s1,Sembungan adalah sebuah desa yang terletak di ...,sebuah,sebuah,DET,B--,{'PronType': 'Ind'},4,det,{'MorphInd': '^sebuah<b>_B--$'}
3,train-s1,Sembungan adalah sebuah desa yang terletak di ...,desa,desa,NOUN,NSD,{'Number': 'Sing'},0,root,{'MorphInd': '^desa<n>_NSD$'}
4,train-s1,Sembungan adalah sebuah desa yang terletak di ...,yang,yang,PRON,S--,{'PronType': 'Rel'},6,nsubj:pass,{'MorphInd': '^yang<s>_S--$'}
5,train-s1,Sembungan adalah sebuah desa yang terletak di ...,terletak,terletak,VERB,VSP,"{'Number': 'Sing', 'Voice': 'Pass'}",4,acl,{'MorphInd': '^ter+letak<n>_VSP$'}
6,train-s1,Sembungan adalah sebuah desa yang terletak di ...,di,di,ADP,R--,-,8,case,{'MorphInd': '^di<r>_R--$'}
7,train-s1,Sembungan adalah sebuah desa yang terletak di ...,kecamatan,kecamatan,NOUN,NSD,{'Number': 'Sing'},6,obl,{'MorphInd': '^ke+camat<n>+an_NSD$'}
8,train-s1,Sembungan adalah sebuah desa yang terletak di ...,Kejajar,kejajar,PROPN,X--,-,8,flat,"{'SpaceAfter': 'No', 'MorphInd': '^kejajar<x>_..."
9,train-s1,Sembungan adalah sebuah desa yang terletak di ...,",",",",PUNCT,Z--,-,8,punct,"{'MorphInd': '^,<z>_Z--$'}"


# Data Annotator

## A. Convert From CoNLL-U to CSV (UD-Indonesian_GSD)

### 1. Set Path

In [94]:
# Direktori CSV untuk Morfologi Afiksasi (non-punct)
OUTPUT_DIR_MORFOLOGI = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\anotasi\all_csv_ud-indonesian-gsd\morfologi_afiksasi"
 
# Direktori CSV untuk Negation Scope Detection (dengan punct, hanya kalimat negasi)
OUTPUT_DIR_NEGASI    = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\anotasi\all_csv_ud-indonesian-gsd\negation_scope_detection"

### 2. Fungsi Parse dan Konversi

In [ ]:
# =========================
# PARSE CONLLU → DATAFRAME
# =========================
 
def parse_conllu_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return conllu.parse(f.read())
 
def conllu_to_dataframe(sentences, exclude_punct=False, only_negation=False):
    rows = []
 
    for kalimat in sentences:
        sent_id = kalimat.metadata.get('sent_id', '')
        teks    = kalimat.metadata.get('text', '')
 
        # ── Filter kalimat negasi (Polarity=Neg di feats) ──────────
        if only_negation:
            ada_negasi = False
            for token in kalimat:
                if (isinstance(token['id'], int)
                        and token['feats']
                        and "'Polarity': 'Neg'" in str(token['feats'])):
                    ada_negasi = True
                    break
            if not ada_negasi:
                continue
        # ────────────────────────────────────────────────────────────
 
        for token in kalimat:
            if not isinstance(token['id'], int):
                continue
            if exclude_punct and token['upos'] == 'PUNCT':
                continue
 
            rows.append({
                'sent_id'  : sent_id,
                'teks'     : teks,
                'token_id' : token['id'],
                'token'    : token['form'],
                'lemma'    : token['lemma'],
                'upos'     : token['upos'],
                'xpos'     : token['xpos'],
                'feats'    : str(token['feats']) if token['feats'] else '_',
                'head'     : token['head'],
                'deprel'   : token['deprel'],
                'misc'     : str(token['misc']) if token['misc'] else '_',
            })
 
    return pd.DataFrame(rows)
 
 
splits = {
    'train' : PATH_GSD_TRAIN,
    'dev'   : PATH_GSD_DEV,
    'test'  : PATH_GSD_TEST,
}
 
raw_sentences = {split: parse_conllu_file(path) for split, path in splits.items()}

### 3. Menyimpan Data Keseluruhan Morfologi Afiksasi (non tanda baca)

In [106]:
df_morfologi = {}
rows_morf    = []
 
print("=== STATISTIK TOKEN PER SPLIT — MORFOLOGI AFIKSASI (TANPA TANDA BACA) ===\n")
 
for split_name, sentences in raw_sentences.items():
    df = conllu_to_dataframe(sentences, exclude_punct=True, only_negation=False)
    df_morfologi[split_name] = df
    rows_morf.append({
        'Split'   : split_name,
        'Kalimat' : len(sentences),
        'Token (exc. PUNCT)'   : len(df),
    })
 
df_stat_morf = pd.DataFrame(rows_morf)
df_stat_morf.loc[len(df_stat_morf)] = [
    'TOTAL',
    df_stat_morf['Kalimat'].sum(),
    df_stat_morf['Token (exc. PUNCT)'].sum(),
]
display(df_stat_morf.reset_index(drop=True))
 
# Simpan
os.makedirs(OUTPUT_DIR_MORFOLOGI, exist_ok=True)
total_tersimpan_morf = 0
 
print("\nStatus penyimpanan (Morfologi Afiksasi):")
for split_name, df in df_morfologi.items():
    out_path = os.path.join(OUTPUT_DIR_MORFOLOGI, f"all_ud-indo-gsd_morfologi_{split_name}.csv")
    try:
        df.to_csv(out_path, index=False, encoding='utf-8-sig')
        status = '✅ BERHASIL'
        total_tersimpan_morf += len(df)
    except Exception as e:
        status = f'❌ GAGAL ({e})'
    print(f"  [{status}] all_ud-indo-gsd_morfologi_{split_name}.csv  →  {len(df):,} token")
 
print(f"\nTotal token tersimpan : {total_tersimpan_morf:,} token")

=== STATISTIK TOKEN PER SPLIT — MORFOLOGI AFIKSASI (TANPA TANDA BACA) ===



,Split,Kalimat,Token (exc. PUNCT)
0,train,4477,82963
1,dev,559,10676
2,test,557,10056
3,TOTAL,5593,103695



Status penyimpanan (Morfologi Afiksasi):
  [✅ BERHASIL] all_ud-indo-gsd_morfologi_train.csv  →  82,963 token
  [✅ BERHASIL] all_ud-indo-gsd_morfologi_dev.csv  →  10,676 token
  [✅ BERHASIL] all_ud-indo-gsd_morfologi_test.csv  →  10,056 token

Total token tersimpan : 103,695 token


### 4. Menyimpan Data Keseluruhan Negation Scope Detection (dengan tanda baca + hanya kalimat negasi)

In [104]:
df_negasi = {}
rows_neg  = []
 
print("=== STATISTIK TOKEN PER SPLIT — NEGATION SCOPE DETECTION (DENGAN TANDA BACA) ===\n")
print("Filter  : hanya kalimat yang mengandung token dengan feats Polarity=Neg\n")
 
for split_name, sentences in raw_sentences.items():
    # Total kalimat sebelum filter (untuk menghitung persentase)
    total_kalimat = len(sentences)
 
    df = conllu_to_dataframe(sentences, exclude_punct=False, only_negation=True)
    df_negasi[split_name] = df
 
    # Hitung jumlah kalimat negasi (unik berdasarkan sent_id)
    kalimat_negasi = df['sent_id'].nunique() if not df.empty else 0
 
    # Hitung total token negasi (Polarity=Neg) di semua kalimat negasi
    token_negasi = df['feats'].apply(
        lambda f: "'Polarity': 'Neg'" in str(f)
    ).sum() if not df.empty else 0
 
    rows_neg.append({
        'Split'              : split_name,
        'Total Kalimat'      : total_kalimat,
        'Kalimat Negasi'     : kalimat_negasi,
        'Token (incl. PUNCT)': len(df),
        'Token Negasi'       : int(token_negasi),
    })
 
df_stat_neg = pd.DataFrame(rows_neg)
df_stat_neg.loc[len(df_stat_neg)] = [
    'TOTAL',
    df_stat_neg['Total Kalimat'].sum(),
    df_stat_neg['Kalimat Negasi'].sum(),
    df_stat_neg['Token (incl. PUNCT)'].sum(),
    df_stat_neg['Token Negasi'].sum(),
]
display(df_stat_neg.reset_index(drop=True))
 
# Simpan
os.makedirs(OUTPUT_DIR_NEGASI, exist_ok=True)
total_tersimpan_neg = 0
 
print("\nStatus penyimpanan (Negation Scope Detection):")
for split_name, df in df_negasi.items():
    out_path = os.path.join(OUTPUT_DIR_NEGASI, f"all_ud-indo-gsd_negasi_{split_name}.csv")
    try:
        df.to_csv(out_path, index=False, encoding='utf-8-sig')
        status = '✅ BERHASIL'
        total_tersimpan_neg += len(df)
    except Exception as e:
        status = f'❌ GAGAL ({e})'
    print(f"  [{status}] all_ud-indo-gsd_negasi_{split_name}.csv  →  {len(df):,} token")
 
print(f"\nTotal token tersimpan : {total_tersimpan_neg:,} token")

=== STATISTIK TOKEN PER SPLIT — NEGATION SCOPE DETECTION (DENGAN TANDA BACA) ===

Filter  : hanya kalimat yang mengandung token dengan feats Polarity=Neg



,Split,Total Kalimat,Kalimat Negasi,Token (incl. PUNCT),Token Negasi
0,train,4477,405,10689,445
1,dev,559,46,1244,52
2,test,557,44,1193,46
3,TOTAL,5593,495,13126,543



Status penyimpanan (Negation Scope Detection):
  [✅ BERHASIL] all_ud-indo-gsd_negasi_train.csv  →  10,689 token
  [✅ BERHASIL] all_ud-indo-gsd_negasi_dev.csv  →  1,244 token
  [✅ BERHASIL] all_ud-indo-gsd_negasi_test.csv  →  1,193 token

Total token tersimpan : 13,126 token


## B. Pembuatan Sample

### 1. Affixtional Morphology Classification (Token Level)

#### a. Perhitungan Jumlah Sample yang Diambil (Formula Taro Yamane)

Formula Taro Yamane digunakan untuk menentukan jumlah sampel dari populasi yang telah diketahui dengan mempertimbangkan batas toleransi error dari sampel yang diambil.

$$n = \frac{N}{1 + N(e)^2}$$

**Keterangan:**
- $n$ = Jumlah sampel yang dibutuhkan
- $N$ = Jumlah populasi keseluruhan
- $e$ = Batas toleransi kesalahan kesalahan sample *(margin of error)*

In [110]:
# ========================
# DIRECTORY SAMPLING DATA
# ========================

OUTPUT_SAMPLING_MORF_CSV   = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\anotasi\morfologi_afiksasi\a_pre_anotasi_sampel\csv"
OUTPUT_SAMPLING_MORF_EXCEL = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\anotasi\morfologi_afiksasi\a_pre_anotasi_sampel\excel"
 
os.makedirs(OUTPUT_SAMPLING_MORF_CSV,   exist_ok=True)
os.makedirs(OUTPUT_SAMPLING_MORF_EXCEL, exist_ok=True)

In [141]:
# ==============================
# HITUNG YAMANE PER SPLIT
# ==============================

e = 0.05  # margin of error 5%
 
rows_yamane_morf = []
 
print("=== SAMPLE SIZE PER SPLIT — MORFOLOGI AFIKSASI (TANPA TANDA BACA) ===")
 
for split_name, df in df_morfologi.items():
    N = len(df)
    n = round(N / (1 + N * (e ** 2)))
    rows_yamane_morf.append({
        'Split'        : split_name,
        'Populasi (N)' : N,
        'Sample (n)'   : n,
    })
 
df_yamane_morf = pd.DataFrame(rows_yamane_morf)
df_yamane_morf.loc[len(df_yamane_morf)] = [
    'TOTAL',
    df_yamane_morf['Populasi (N)'].sum(),
    df_yamane_morf['Sample (n)'].sum(),
]
display(df_yamane_morf.reset_index(drop=True))
print(f"Margin of error (e) : {e} ({int(e*100)}%)")

=== SAMPLE SIZE PER SPLIT — MORFOLOGI AFIKSASI (TANPA TANDA BACA) ===


,Split,Populasi (N),Sample (n)
0,train,82963,398
1,dev,10676,386
2,test,10056,385
3,TOTAL,103695,1169


Margin of error (e) : 0.05 (5%)


#### b. Pemilihan Sample Secara Random

In [ ]:
# ==============================
# RANDOM SAMPLING & SIMPAN 
# ==============================

ANNOTATION_COLUMNS_MORF = [
    'sent_id', 'teks', 'token_id', 'token', 'lemma',
    'upos', 'xpos', 'feats', 'head', 'deprel', 'misc',
    'tipe_afiks',
]
 
print("=== RANDOM SAMPLING & SIMPAN TEMPLATE ANOTASI — MORFOLOGI AFIKSASI ===")
print()
 
for row in rows_yamane_morf:
    split_name = row['Split']
    n          = row['Sample (n)']
    df         = df_morfologi[split_name]
 
    # Sampling acak
    df_sample = df.sample(n=n, random_state=42).reset_index(drop=True)
 
    # Tambah kolom anotasi kosong
    df_sample['tipe_afiks'] = ''
 
    # Pastikan urutan kolom sesuai template
    df_sample = df_sample[ANNOTATION_COLUMNS_MORF]
 
    path_csv   = os.path.join(OUTPUT_SAMPLING_MORF_CSV,   f"morfologiAfiksasi_sample_{split_name}.csv")
    path_excel = os.path.join(OUTPUT_SAMPLING_MORF_EXCEL, f"morfologiAfiksasi_sample_{split_name}.xlsx")
 
    # Simpan CSV
    try:
        df_sample.to_csv(path_csv, index=False, encoding='utf-8-sig')
        status_csv = '✅ BERHASIL'
    except Exception as ex:
        status_csv = f'❌ GAGAL ({ex})'
 
    # Simpan Excel (lebar kolom disesuaikan)
    try:
        with pd.ExcelWriter(path_excel, engine='openpyxl') as writer:
            df_sample.to_excel(writer, index=False, sheet_name='Anotasi')
 
            ws = writer.sheets['Anotasi']
 
            col_widths = {
                'sent_id'    : 10,
                'teks'       : 60,
                'token_id'   : 5,
                'token'      : 18,
                'lemma'      : 18,
                'upos'       : 12,
                'xpos'       : 15,
                'feats'      : 30,
                'head'       : 10,
                'deprel'     : 15,
                'misc'       : 50,
                'tipe_afiks' : 20,
            }
            for i, col_name in enumerate(ANNOTATION_COLUMNS_MORF, start=1):
                col_letter = ws.cell(row=1, column=i).column_letter
                ws.column_dimensions[col_letter].width = col_widths.get(col_name, 15)
 
        status_excel = '✅ BERHASIL'
    except Exception as ex:
        status_excel = f'❌ GAGAL ({ex})'
 
    print(f"[{split_name}]  {n:,} token acak diambil dari {row['Populasi (N)']:,} token")
    print(f"  CSV   [{status_csv}]")
    print(f"  Excel [{status_excel}]")
    print()
 
print("Status: Proses sampling selesai.")

=== RANDOM SAMPLING & SIMPAN TEMPLATE ANOTASI — MORFOLOGI AFIKSASI ===

[train]  398 token acak diambil dari 82,963 token
  CSV   [✅ BERHASIL]
  Excel [✅ BERHASIL]

[dev]  386 token acak diambil dari 10,676 token
  CSV   [✅ BERHASIL]
  Excel [✅ BERHASIL]

[test]  385 token acak diambil dari 10,056 token
  CSV   [✅ BERHASIL]
  Excel [✅ BERHASIL]

Status: Proses sampling selesai.


### 2. Negation Scope Detection (Sentence Level)

#### a. Konversi Ke Dataframe Level Kalimat

In [122]:
def neg_token_to_sentence_df(df_token):
    """
    Mengkonversi DataFrame token-level kalimat negasi
    ke DataFrame sentence-level (satu baris per kalimat).
 
    Kolom token_negasi berisi semua token ber-Polarity=Neg
    dalam format "token_id:token", dipisah koma.
    Contoh: "3:tidak, 8:bukan"
    """
    rows = []
    for sent_id, grp in df_token.groupby('sent_id', sort=False):
        teks = grp['teks'].iloc[0]
 
        neg_tokens = grp[
            grp['feats'].apply(lambda f: "'Polarity': 'Neg'" in str(f))
        ][['token_id', 'token']]
 
        token_negasi = ', '.join(
            f"{int(r['token_id'])}:{r['token']}"
            for _, r in neg_tokens.iterrows()
        )
 
        rows.append({
            'sent_id'      : sent_id,
            'teks'         : teks,
            'token_negasi' : token_negasi,
        })
    return pd.DataFrame(rows)
 
 
neg_sent_df = {}
 
print("=== STATISTIK KALIMAT NEGASI PER SPLIT ===\n")
 
for split_name, df_tok in df_negasi.items():
    df_sent = neg_token_to_sentence_df(df_tok)
    neg_sent_df[split_name] = df_sent
 
print(f"{'Split':<10} {'Kalimat Negasi':>18}")
print("-" * 30)
for split_name, df_sent in neg_sent_df.items():
    print(f"{split_name:<10} {len(df_sent):>18,}")
print(f"{'TOTAL':<10} {sum(len(d) for d in neg_sent_df.values()):>18,}")

=== STATISTIK KALIMAT NEGASI PER SPLIT ===

Split          Kalimat Negasi
------------------------------
train                     405
dev                        46
test                       44
TOTAL                     495


#### b. Perhitungan Jumlah Sample yang Diambil (Formula Taro Yamane)

Formula Taro Yamane digunakan untuk menentukan jumlah sampel dari populasi yang telah diketahui dengan mempertimbangkan batas toleransi error dari sampel yang diambil.

$$n = \frac{N}{1 + N(e)^2}$$

**Keterangan:**
- $n$ = Jumlah sampel yang dibutuhkan
- $N$ = Jumlah populasi keseluruhan
- $e$ = Batas toleransi kesalahan kesalahan sample *(margin of error)*

In [137]:
# ========================
# DIRECTORY SAMPLING DATA
# ========================

OUTPUT_SAMPLING_NEG_CSV   = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\anotasi\negation_scope_detection\a_pre_anotasi_sampel\csv"
OUTPUT_SAMPLING_NEG_EXCEL = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\anotasi\negation_scope_detection\a_pre_anotasi_sampel\excel"
 
os.makedirs(OUTPUT_SAMPLING_NEG_CSV,   exist_ok=True)
os.makedirs(OUTPUT_SAMPLING_NEG_EXCEL, exist_ok=True)

In [138]:
# ==============================
# HITUNG YAMANE PER SPLIT
# ==============================

e = 0.05  # margin of error 5%
 
rows_yamane_neg = []
 
print("=== SAMPLE SIZE PER SPLIT — NEGATION SCOPE DETECTION (SENTENCE LEVEL) ===")
 
for split_name, df_sent in neg_sent_df.items():
    N = len(df_sent)
    n = round(N / (1 + N * (e ** 2)))
    rows_yamane_neg.append({
        'Split'        : split_name,
        'Populasi (N)' : N,
        'Sample (n)'   : n,
    })
 
df_yamane_neg = pd.DataFrame(rows_yamane_neg)
df_yamane_neg.loc[len(df_yamane_neg)] = [
    'TOTAL',
    df_yamane_neg['Populasi (N)'].sum(),
    df_yamane_neg['Sample (n)'].sum(),
]
display(df_yamane_neg.reset_index(drop=True))
print(f"Margin of error (e) : {e} ({int(e*100)}%)\n")

=== SAMPLE SIZE PER SPLIT — NEGATION SCOPE DETECTION (SENTENCE LEVEL) ===


,Split,Populasi (N),Sample (n)
0,train,405,201
1,dev,46,41
2,test,44,40
3,TOTAL,495,282


Margin of error (e) : 0.05 (5%)



#### b. Pemilihan Sample Secara Random

In [139]:
# ==============================
# RANDOM SAMPLING & SIMPAN 
# ==============================

ANNOTATION_COLUMNS_NEG = [
    'sent_id', 'teks', 'token_id', 'token', 'lemma',
    'upos', 'xpos', 'feats', 'head', 'deprel', 'misc',
    'negasi_scope',
]
 
print("=== RANDOM SAMPLING & SIMPAN TEMPLATE ANOTASI — NEGATION SCOPE DETECTION ===")
print()

total_token_all = 0 
 
for row in rows_yamane_neg:
    split_name = row['Split']
    n          = row['Sample (n)']
    df_tok     = df_negasi[split_name]
 
    all_sent_ids     = df_tok['sent_id'].unique()
    sampled_sent_ids = pd.Series(all_sent_ids).sample(n=n, random_state=42).values
 
    df_sample = df_tok[df_tok['sent_id'].isin(sampled_sent_ids)].copy()
    df_sample  = df_sample.sort_values(['sent_id', 'token_id']).reset_index(drop=True)
 
    df_sample['negasi_scope'] = ''
    df_sample = df_sample[ANNOTATION_COLUMNS_NEG]

    total_token_all += len(df_sample)
 
    path_csv   = os.path.join(OUTPUT_SAMPLING_NEG_CSV,   f"negasiScope_sample_{split_name}.csv")
    path_excel = os.path.join(OUTPUT_SAMPLING_NEG_EXCEL, f"negasiScope_sample_{split_name}.xlsx")
 
    try:
        df_sample.to_csv(path_csv, index=False, encoding='utf-8-sig')
        status_csv = '✅ BERHASIL'
    except Exception as ex:
        status_csv = f'❌ GAGAL ({ex})'
 
    try:
        with pd.ExcelWriter(path_excel, engine='openpyxl') as writer:
            df_sample.to_excel(writer, index=False, sheet_name='Anotasi')
            ws = writer.sheets['Anotasi']
            col_widths = {
                'sent_id'      : 20,
                'teks'         : 60,
                'token_id'     : 12,
                'token'        : 18,
                'lemma'        : 18,
                'upos'         : 12,
                'xpos'         : 15,
                'feats'        : 30,
                'head'         : 10,
                'deprel'       : 15,
                'misc'         : 20,
                'negasi_scope' : 20,
            }
            for i, col_name in enumerate(ANNOTATION_COLUMNS_NEG, start=1):
                col_letter = ws.cell(row=1, column=i).column_letter
                ws.column_dimensions[col_letter].width = col_widths.get(col_name, 15)
        status_excel = '✅ BERHASIL'
    except Exception as ex:
        status_excel = f'❌ GAGAL ({ex})'
 
    print(f"[{split_name}]  {n:,} kalimat acak diambil dari {row['Populasi (N)']:,} kalimat negasi")
    print(f"  Token tersimpan : {len(df_sample):,} token")
    print(f"  CSV   [{status_csv}]")
    print(f"  Excel [{status_excel}]")
    print()
 
print(f"Total token seluruh split : {total_token_all:,} token") 
print("Status: Proses sampling & pembuatan template anotasi selesai.")

=== RANDOM SAMPLING & SIMPAN TEMPLATE ANOTASI — NEGATION SCOPE DETECTION ===

[train]  201 kalimat acak diambil dari 405 kalimat negasi
  Token tersimpan : 5,414 token
  CSV   [✅ BERHASIL]
  Excel [✅ BERHASIL]

[dev]  41 kalimat acak diambil dari 46 kalimat negasi
  Token tersimpan : 1,057 token
  CSV   [✅ BERHASIL]
  Excel [✅ BERHASIL]

[test]  40 kalimat acak diambil dari 44 kalimat negasi
  Token tersimpan : 1,043 token
  CSV   [✅ BERHASIL]
  Excel [✅ BERHASIL]

Total token seluruh split : 7,514 token
Status: Proses sampling & pembuatan template anotasi selesai.
